In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, hashlib, subprocess
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
for m in ['config','features_cic']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config, features_cic as fc
import numpy as np, pandas as pd
print('ready:', os.getcwd())


Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [ ]:
# =============================================================================
# Cell 2 - labels per realization + randomized-APS / Mondrian / coverage
# functions (preregistration 7). Two classes, column order [Benign, DoS].
# =============================================================================
cic = pd.read_parquet(config.INTERIM_DIR/'cicids2017_primary.parquet')
wed = cic[cic['day']=='wednesday'].reset_index(drop=True)
wed = wed[wed['label'].isin(['DoS','Benign'])].reset_index(drop=True)

BEN, DOS = 0, 1
PROBS_DIR = config.DATA_DIR / 'cic_probs'
REALIZATIONS = ['R1_holdout_Slowhttptest','R2_holdout_Slowloris','R3_holdout_GoldenEye',
                'R4_holdout_Slowloris_Slowhttptest','R5_holdout_GoldenEye_Slowloris']

def dseed(*parts):
    return int(hashlib.sha256('|'.join(map(str, parts)).encode()).hexdigest(), 16) % (2**32)

def labels_for(idx):
    return (wed.loc[idx,'label'].to_numpy() == 'DoS').astype(int)

def aps_scores_all(P, rng):
    order = np.argsort(-P, axis=1); sp = np.take_along_axis(P, order, 1)
    cum = np.cumsum(sp, 1); U = rng.random(len(P))[:, None]
    sorted_scores = cum - (1 - U) * sp
    scores = np.empty_like(P); np.put_along_axis(scores, order, sorted_scores, 1)
    return scores

def qhat(true_scores, alpha):
    n = len(true_scores)
    if n < 1: return np.inf
    lvl = min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    return float(np.quantile(true_scores, lvl, method='higher'))

def coverage(eval_scores, y_eval, cal_scores_all, y_cal, alpha):
    tc = cal_scores_all[np.arange(len(y_cal)), y_cal]
    q = {c: qhat(tc[y_cal == c], alpha) for c in (BEN, DOS)}
    inset = np.column_stack([eval_scores[:, c] <= q[c] for c in (BEN, DOS)])
    cov = {c: (float(inset[y_eval == c, c].mean()) if (y_eval == c).any() else np.nan) for c in (BEN, DOS)}
    ss = inset.sum(1)
    return cov, {c: (float(ss[y_eval == c].mean()) if (y_eval == c).any() else np.nan) for c in (BEN, DOS)}

print('functions ready; realizations:', len(REALIZATIONS))


functions ready; realizations: 5


In [ ]:
# =============================================================================
# Cell 3 - three-protocol coverage over matched draws (section 8.1). For each
# (realization, seed, arch): R matched draws of D_eval / T_cal / S_cal, then
# REC (calibrate on D_eval), TSC (on T_cal), SHC (on S_cal), all evaluated on
# D_eval. Coverage per class + set size, averaged over draws.
# =============================================================================
ALPHAS = [config.ALPHA_PRIMARY] + config.ALPHA_SENSITIVITY
R = getattr(config, 'N_MATCHED_DRAWS', 10)
rows = []

for name in REALIZATIONS:
    sp_idx = np.load(config.PROC_DIR/f'cic_{name}_srcpool_idx.npy')
    tg_idx = np.load(config.PROC_DIR/f'cic_{name}_target_idx.npy')
    y_sp_full, y_tg_full = labels_for(sp_idx), labels_for(tg_idx)
    m = min(len(sp_idx), len(tg_idx)//2)

    for seed in config.SEEDS:
        for arch in ['rf','xgb','mlp']:
            f = PROBS_DIR/f'{name}__{arch}__seed{seed}.npz'
            if not f.exists(): continue
            d = np.load(f); Psp, Ptg = d['srcpool'], d['target']
            acc = {}
            for draw in range(R):
                rng = np.random.default_rng(dseed(name, seed, arch, draw))
                tp = rng.permutation(len(tg_idx)); de_i, tc_i = tp[:m], tp[m:2*m]
                sc_i = rng.permutation(len(sp_idx))[:m]
                P_de, y_de = Ptg[de_i], y_tg_full[de_i]
                P_tc, y_tc = Ptg[tc_i], y_tg_full[tc_i]
                P_sc, y_sc = Psp[sc_i], y_sp_full[sc_i]

                es = aps_scores_all(P_de, np.random.default_rng(dseed(name,seed,arch,draw,'e')))
                cal = {'REC': (aps_scores_all(P_de, np.random.default_rng(dseed(name,seed,arch,draw,'rc'))), y_de),
                       'TSC': (aps_scores_all(P_tc, np.random.default_rng(dseed(name,seed,arch,draw,'tc'))), y_tc),
                       'SHC': (aps_scores_all(P_sc, np.random.default_rng(dseed(name,seed,arch,draw,'sc'))), y_sc)}
                for proto,(cs,yc) in cal.items():
                    for a in ALPHAS:
                        cov, ss = coverage(es, y_de, cs, yc, a)
                        for c in (BEN, DOS):
                            acc.setdefault((proto,c,a,'cov'),[]).append(cov[c])
                            acc.setdefault((proto,c,a,'ss'),[]).append(ss[c])
            for (proto,c,a,kind),vals in acc.items():
                if kind!='cov': continue
                rows.append({'dataset':'cicids2017','realization':name,'seed':seed,'arch':arch,
                             'protocol':proto,'class':'DoS' if c==DOS else 'Benign','alpha':a,
                             'coverage':round(float(np.nanmean(vals)),4),
                             'set_size':round(float(np.nanmean(acc[(proto,c,a,'ss')])),4),
                             'nominal':round(1-a,4)})
    print('done', name)

cov = pd.DataFrame(rows)
cov.to_csv(config.REPORTS_DIR/'coverage_primary_cicids2017.csv', index=False)
print('\ncoverage rows:', len(cov))


done R1_holdout_Slowhttptest
done R2_holdout_Slowloris
done R3_holdout_GoldenEye
done R4_holdout_Slowloris_Slowhttptest
done R5_holdout_GoldenEye_Slowloris

coverage rows: 2700


In [ ]:
# =============================================================================
# Cell 4 - FOCAL RESULT: DoS coverage by protocol at the primary alpha and the
# TSC-vs-SHC gap (criterion 3). Overall and per realization against S_sup.
# =============================================================================
prim = cov[(cov['class']=='DoS') & (cov['alpha']==config.ALPHA_PRIMARY)]
overall = prim.groupby('protocol')['coverage'].mean().round(4)
print('DoS focal coverage at alpha =', config.ALPHA_PRIMARY, ' (nominal', round(1-config.ALPHA_PRIMARY,3),'):')
print(overall.to_string())

gap = float(overall.get('TSC',np.nan) - overall.get('SHC',np.nan))
print(f'\nfocal gap  TSC - SHC = {gap:+.4f}   (criterion 3 bar: >= 0.05)')
print('criterion 3 (focal gap >= 5pp):', 'PASS' if gap >= 0.05 else 'NOT MET')

per = prim.groupby(['realization','protocol'])['coverage'].mean().unstack('protocol').round(4)
shift = pd.read_csv(config.REPORTS_DIR/'ladder_shift_measures_cicids2017.csv').set_index('realization')
per['S_sup'] = shift['S_sup']; per['held_out_dos_mass'] = shift['held_out_dos_mass']
per['gap_TSC_SHC'] = (per['TSC'] - per['SHC']).round(4)
per = per.sort_values('S_sup')
print('\nper realization (sorted by held-out support):')
print(per.to_string())
per.to_csv(config.REPORTS_DIR/'coverage_focal_contrast_cicids2017.csv')

verdict = {'dataset':'cicids2017','environment':'wednesday_within_day','focal_class':'DoS',
           'alpha_primary':config.ALPHA_PRIMARY,'nominal':round(1-config.ALPHA_PRIMARY,4),
           'DoS_coverage_by_protocol':{k:float(v) for k,v in overall.items()},
           'focal_gap_TSC_minus_SHC':round(gap,4),'criterion3_bar':0.05,
           'criterion3':'PASS' if gap>=0.05 else 'NOT MET'}
(config.REPORTS_DIR/'cicids2017_focal_verdict.json').write_text(json.dumps(verdict,indent=2))
print('\n', json.dumps(verdict, indent=2))


DoS focal coverage at alpha = 0.05  (nominal 0.95 ):
protocol
REC    0.9875
SHC    0.6039
TSC    0.9876

focal gap  TSC - SHC = +0.3837   (criterion 3 bar: >= 0.05)
criterion 3 (focal gap >= 5pp): PASS

per realization (sorted by held-out support):
protocol                              REC     SHC     TSC   S_sup  held_out_dos_mass  gap_TSC_SHC
realization                                                                                      
R1_holdout_Slowhttptest            0.9536  0.8548  0.9547  0.0112               1741       0.0999
R2_holdout_Slowloris               1.0000  0.4978  1.0000  0.0255               3998       0.5022
R4_holdout_Slowloris_Slowhttptest  1.0000  0.2923  1.0000  0.0361               5739       0.7077
R3_holdout_GoldenEye               0.9837  0.7395  0.9833  0.0471               7567       0.2438
R5_holdout_GoldenEye_Slowloris     1.0000  0.6352  1.0000  0.0703              11565       0.3648

 {
  "dataset": "cicids2017",
  "environment": "wednesday_within

In [ ]:
# =============================================================================
# Cell 5 - recorded feature-separability audit (answers the leak question on the
# record) and commit.
# =============================================================================
from sklearn.metrics import roc_auc_score
FCOLS = fc.feature_cols(wed); yb = (wed['label']=='DoS').astype(int).to_numpy()
aud = []
for c in FCOLS:
    x = pd.to_numeric(wed[c], errors='coerce').to_numpy(float); x = np.where(np.isfinite(x), x, np.nan)
    good = ~np.isnan(x)
    if good.sum()==0 or np.nanstd(x[good])==0: auc=0.5
    else: auc = roc_auc_score(yb, np.where(np.isnan(x), np.nanmedian(x), x))
    aud.append({'feature':c,'auc':round(float(auc),4),'separation':round(abs(auc-0.5),4)})
audit = pd.DataFrame(aud).sort_values('separation', ascending=False)
audit.to_csv(config.REPORTS_DIR/'cicids2017_feature_separability.csv', index=False)
print('top single-feature separability (max auc %.3f; none is an identifier at ~1.0):' % audit['auc'].max())
print(audit.head(8).to_string(index=False))

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb13: CIC three-protocol conformal coverage, DoS focal gap, feature-separability audit')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


top single-feature separability (max auc 0.984; none is an identifier at ~1.0):
               feature    auc  separation
     Packet Length Std 0.9838      0.4838
Packet Length Variance 0.9838      0.4838
 Bwd Packet Length Std 0.9798      0.4798
     Packet Length Max 0.9785      0.4785
 Bwd Packet Length Max 0.9674      0.4674
    Packet Length Mean 0.9670      0.4670
       Avg Packet Size 0.9670      0.4670
  Avg Bwd Segment Size 0.9663      0.4663
[main 917f518] nb13: CIC three-protocol conformal coverage, DoS focal gap, feature-separability audit
 6 files changed, 2806 insertions(+), 1 deletion(-)
 create mode 100644 notebooks/13_cic_conformal_coverage.ipynb
 create mode 100644 reports/cicids2017_feature_separability.csv
 create mode 100644 reports/cicids2017_focal_verdict.json
 create mode 100644 reports/coverage_focal_contrast_cicids2017.csv
 create mode 100644 reports/coverage_primary_cicids2017.csv
Branch 'main' set up to track remote branch 'main' from 'origin'.
To https://

In [2]:
import shutil
from pathlib import Path

UGR = Path('/content/drive/MyDrive/NIDS_Datasets/ugr16')
print('ugr16 folder exists:', UGR.exists())
if UGR.exists():
    items = [p for p in UGR.glob('**/*') if p.is_file()]
    print('files present:', len(items))
    for p in sorted(items)[:40]:
        print(f'  {p.relative_to(UGR)}   {p.stat().st_size/1e6:.1f} MB')

for label, path in [('Drive', '/content/drive/MyDrive'), ('Colab local disk', '/content')]:
    total, used, free = shutil.disk_usage(path)
    print(f'{label}: {free/1e9:.1f} GB free of {total/1e9:.1f} GB')

ugr16 folder exists: False
Drive: 210.0 GB free of 242.5 GB
Colab local disk: 221.0 GB free of 242.5 GB
